In [8]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

Matplotlib is building the font cache; this may take a moment.


In [10]:
df=pd.read_csv("final_crypto.csv")

In [11]:
df=df[df["type"]!="XRP"]
df=df[df["type"]!="BTC"]

In [12]:
train_coins = ['ETH', 'BNB', 'SOL', 'ADA']
test_coins = ['DOGE', 'AVAX']

train_df = df[df['type'].isin(train_coins)].copy()
test_df  = df[df['type'].isin(test_coins)].copy()

print("Coins in train:", train_df['type'].unique())
print("Coins in test:", test_df['type'].unique())

Coins in train: ['ETH' 'BNB' 'SOL']
Coins in test: []


In [13]:
df.columns.tolist()

['date',
 'num_trades',
 'taker_buy_volume',
 'type',
 'candle_body',
 'high_low_range',
 'ma7_ratio',
 'ma30_ratio',
 'moving_1d',
 'volatility_7',
 'volumespike',
 'z_score',
 'taker_buy_ratio',
 'price_position',
 'momentum_7',
 'taker_sell_ratio',
 'negative_momentum',
 'down_days_7',
 'volatility_14',
 'volatility_3',
 'label']

In [14]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder





drop_cols = [
    'label', 'date', 'type',
    'down_days_7',
    'taker_buy_volume',
    'z_score','taker_buy_ratio','taker_buy_ratio', 'negative_momentum',
]

X_train = train_df.drop(columns=drop_cols)
y_train = train_df['label']

X_test = test_df.drop(columns=drop_cols)
y_test = test_df['label']


model = RandomForestClassifier(
    n_estimators=700,
    max_depth=13,
    min_samples_split=5,
    min_samples_leaf=3,
    max_features='sqrt',
    criterion='entropy',
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
print(f"Train: {len(X_train)} | Test: {len(X_test)}")

model.fit(X_train, y_train)






test_preds = model.predict(X_test)
print("\n--- Test Results ---")
print(classification_report(y_test, test_preds, target_names=['Bigdown', 'Bigup','Stable']))


cm = confusion_matrix(y_test, test_preds)
sns.heatmap(cm, annot=True, fmt='d', xticklabels=['Bigdown', 'Bigup','Stable'],
                                     yticklabels=['Bigdown', 'Bigup','Stable'])
plt.title("Confusion Matrix")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.show()


feat_imp = pd.Series(model.feature_importances_, index=X_train.columns)
feat_imp.sort_values().plot(kind='barh', figsize=(8, 6))
plt.title("Feature Importance")
plt.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'xgboost'

In [ ]:

# Check train performance too
train_preds = model.predict(X_train)
print("--- Train Results ---")
print(classification_report(y_train, train_preds, target_names=['Bigdown', 'Bigup','Stable']))

print("--- Test Results ---")
print(classification_report(y_test, test_preds, target_names=['Bigdown', 'Bigup','Stable']))

In [ ]:
print(y_train.unique())
print(sorted(y_train.unique()))

In [15]:
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
import pandas as pd
import numpy as np

drop_cols = [
    'label', 'date', 'type',
    'down_days_7',
    'taker_buy_volume',
    'z_score','taker_buy_ratio','taker_sell_ratio', 'negative_momentum',
]

X = df.drop(columns=drop_cols)
y = df['label']

groups = df['type']

logo = LeaveOneGroupOut()

results = []

for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups), 1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    test_coin = groups.iloc[test_idx].unique()[0]

    model = RandomForestClassifier(
        n_estimators=700,
        max_depth=13,
        min_samples_split=5,
        min_samples_leaf=3,
        criterion='entropy',
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    acc = accuracy_score(y_test, preds)
    macro_f1 = f1_score(y_test, preds, average='macro')
    weighted_f1 = f1_score(y_test, preds, average='weighted')

    results.append({
        'test_coin': test_coin,
        'accuracy': acc,
        'macro_f1': macro_f1,
        'weighted_f1': weighted_f1
    })

    print(f"\n===== Fold {fold} | Test coin: {test_coin} =====")
    print(classification_report(y_test, preds))


===== Fold 1 | Test coin: BNB =====
              precision    recall  f1-score   support

     Bigdown       0.42      0.22      0.29      4164
       Bigup       0.44      0.30      0.35      4165
      Stable       0.49      0.78      0.60      5548

    accuracy                           0.47     13877
   macro avg       0.45      0.43      0.42     13877
weighted avg       0.45      0.47      0.43     13877


===== Fold 2 | Test coin: ETH =====
              precision    recall  f1-score   support

     Bigdown       0.35      0.52      0.42      4166
       Bigup       0.40      0.28      0.33      4165
      Stable       0.54      0.47      0.50      5546

    accuracy                           0.43     13877
   macro avg       0.43      0.42      0.42     13877
weighted avg       0.44      0.43      0.42     13877


===== Fold 3 | Test coin: SOL =====
              precision    recall  f1-score   support

     Bigdown       0.35      0.33      0.34      3759
       Bigup      

In [16]:
results_df = pd.DataFrame(results)

print(results_df)

print("\nAverage Results:")
print(results_df[['accuracy', 'macro_f1', 'weighted_f1']].mean())

  test_coin  accuracy  macro_f1  weighted_f1
0       BNB  0.467897  0.415067     0.433603
1       ETH  0.425957  0.416269     0.424592
2       SOL  0.360555  0.336600     0.325818

Average Results:
accuracy       0.418136
macro_f1       0.389312
weighted_f1    0.394671
dtype: float64


# SVM


In [17]:

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, classification_report, f1_score, confusion_matrix

In [18]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

scaler = StandardScaler()

drop_cols = [
    'label', 'date', 'type',
    'down_days_7',
    'taker_buy_volume',
    'z_score',
    'taker_buy_ratio',
    'taker_sell_ratio',
    'negative_momentum',
]

X_train = train_df.drop(columns=drop_cols)
y_train = train_df['label']

X_test = test_df.drop(columns=drop_cols)
y_test = test_df['label']

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

clf = SVC(
    kernel='rbf',
    C=50,
    gamma='scale',
    class_weight='balanced',
    random_state=42
)

clf.fit(X_train_scaled, y_train)

predictions = clf.predict(X_test_scaled)

ValueError: Found array with 0 sample(s) (shape=(0, 12)) while a minimum of 1 is required by StandardScaler.

In [ ]:
svm_test_predictions = clf.predict(X_test_scaled)
svm_train_predictions = clf.predict(X_train_scaled)

print(f"Accuracy: {accuracy_score(y_test, svm_test_predictions):.4f}")
print("\n--- SVM Test Results ---")
print(classification_report(y_test, svm_test_predictions, target_names=['Bigdown', 'Bigup', 'Stable']))

cm_svm = confusion_matrix(y_test, svm_test_predictions)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_svm, annot=True, fmt='d', cmap='Blues', xticklabels=['Bigdown', 'Bigup','Stable'], yticklabels=['Bigdown', 'Bigup','Stable'])
plt.title("SVM Confusion Matrix")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.show()

print("\n--- SVM Train Results ---")
print(classification_report(y_train, svm_train_predictions, target_names=['Bigdown', 'Bigup', 'Stable']))

In [ ]:
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report
import pandas as pd
import numpy as np

drop_cols = [
    'label', 'date', 'type',
    'down_days_7',
    'taker_buy_volume',
    'z_score',
    'taker_buy_ratio',
    'taker_sell_ratio',
    'negative_momentum',
]

X = df.drop(columns=drop_cols)
y = df['label']

groups = df['type']

logo = LeaveOneGroupOut()

results = []

for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups), 1):

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    test_coin = groups.iloc[test_idx].unique()[0]

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    clf = SVC(
    kernel='rbf',
    C=50,
    gamma='scale',
    class_weight='balanced',
    random_state=42
    )

    clf.fit(X_train_scaled, y_train)

    preds = clf.predict(X_test_scaled)

    acc = accuracy_score(y_test, preds)
    macro_f1 = f1_score(y_test, preds, average='macro')
    weighted_f1 = f1_score(y_test, preds, average='weighted')

    results.append({
        'test_coin': test_coin,
        'accuracy': acc,
        'macro_f1': macro_f1,
        'weighted_f1': weighted_f1
    })

    print(f"\n===== Fold {fold} | Test coin: {test_coin} =====")
    print(classification_report(y_test, preds))

results_df = pd.DataFrame(results)

print("\nResults per coin:")
print(results_df)

print("\nAverage results:")
print(results_df[['accuracy', 'macro_f1', 'weighted_f1']].mean())

In [ ]:
results_df = pd.DataFrame(results)

print(results_df)

print("\nAverage Results:")
print(results_df[['accuracy', 'macro_f1', 'weighted_f1']].mean())

# logistic regression

In [ ]:
from sklearn.linear_model import LogisticRegression
drop_cols = [
    'label', 'date', 'type',
    'down_days_7',
    'taker_buy_volume',
    'z_score','taker_buy_ratio','taker_sell_ratio', 'negative_momentum',
]
X_train = train_df.drop(columns=drop_cols)
y_train = train_df['label']

X_test = test_df.drop(columns=drop_cols)
y_test = test_df['label']
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_reg = LogisticRegression(
        multi_class='multinomial',
        solver='saga',
        class_weight='balanced',
        max_iter=5000,
        random_state=42,
        n_jobs=-1
    )
print(f"Training Logistic Regression... Train: {len(X_train_scaled)}")
log_reg.fit(X_train_scaled, y_train)

In [ ]:
log_test_predictions = log_reg.predict(X_test_scaled)
log_train_predictions = log_reg.predict(X_train_scaled)

print("\n--- Logistic Regression Test Results ---")
print(classification_report(y_test, log_test_predictions, target_names=['Bigdown', 'Bigup', 'Stable']))

cm_log = confusion_matrix(y_test, log_test_predictions)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_log, annot=True, fmt='d', cmap='Greens', xticklabels=['Bigdown', 'Bigup','Stable'], yticklabels=['Bigdown', 'Bigup','Stable'])
plt.title("Logistic Regression Confusion Matrix")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.show()

importance = np.mean(np.abs(log_reg.coef_), axis=0)
feat_imp_log = pd.Series(importance, index=X_train.columns)
feat_imp_log.sort_values().plot(kind='barh', figsize=(8, 6), color='skyblue')
plt.title("Logistic Regression Feature Importance (Abs Coeff)")
plt.tight_layout()
plt.show()

print("--- Logistic Regression Train Results ---")
print(classification_report(y_train, log_train_predictions, target_names=['Bigdown', 'Bigup', 'Stable']))

In [ ]:
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report
import pandas as pd
import numpy as np

drop_cols = [
    'label', 'date', 'type',
    'down_days_7',
    'taker_buy_volume',
    'z_score',
    'taker_buy_ratio',
    'taker_sell_ratio',
    'negative_momentum',
]

X = df.drop(columns=drop_cols)
y = df['label']
groups = df['type']

logo = LeaveOneGroupOut()
results = []

for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups), 1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    test_coin = groups.iloc[test_idx].unique()[0]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    log_reg = LogisticRegression(
        multi_class='multinomial',
        solver='saga',
        class_weight='balanced',
        max_iter=5000,
        random_state=42,
        n_jobs=-1
    )

    log_reg.fit(X_train_scaled, y_train)

    preds = log_reg.predict(X_test_scaled)

    acc = accuracy_score(y_test, preds)
    macro_f1 = f1_score(y_test, preds, average='macro')
    weighted_f1 = f1_score(y_test, preds, average='weighted')

    results.append({
        'test_coin': test_coin,
        'accuracy': acc,
        'macro_f1': macro_f1,
        'weighted_f1': weighted_f1
    })

    print(f"\n===== Fold {fold} | Test coin: {test_coin} =====")
    print(classification_report(y_test, preds))

In [ ]:
results_df = pd.DataFrame(results)

print(results_df)

print("\nAverage Results:")
print(results_df[['accuracy', 'macro_f1', 'weighted_f1']].mean())

In [ ]:
XGboost

In [ ]:
import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.model_selection import LeaveOneGroupOut
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

In [ ]:
drop_cols = [
    'label', 'date', 'type',
    'down_days_7',
    'taker_buy_volume',
    'z_score', 'taker_buy_ratio', 'taker_sell_ratio', 'negative_momentum',
]
X_train = train_df.drop(columns=drop_cols)
y_train = train_df['label']
X_test  = test_df.drop(columns=drop_cols)
y_test  = test_df['label']

In [ ]:
df = df.sort_values('date').reset_index(drop=True)

split_idx = int(len(df) * 0.8)
train_df  = df.iloc[:split_idx].copy()
test_df   = df.iloc[split_idx:].copy()

print(f"train_df: {train_df.shape} | test_df: {test_df.shape}")

drop_cols_all = [
    'label', 'date', 'type',
    'down_days_7', 'taker_buy_volume',
    'z_score', 'taker_buy_ratio', 'taker_sell_ratio', 'negative_momentum',
]
drop_cols = [c for c in drop_cols_all if c in train_df.columns]

X_train = train_df.drop(columns=drop_cols)
y_train = train_df['label']
X_test  = test_df.drop(columns=drop_cols)
y_test  = test_df['label']

X_train = X_train.dropna()
y_train = y_train.loc[X_train.index]
X_test  = X_test.dropna()
y_test  = y_test.loc[X_test.index]

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")

from sklearn.preprocessing import LabelEncoder
from collections import Counter
import numpy as np
import xgboost as xgb

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)
print(f"Classes: {le.classes_}")

counts  = Counter(y_train_enc)
total   = len(y_train_enc)
weights = {cls: total / (len(counts) * cnt) for cls, cnt in counts.items()}
sample_weights = np.array([weights[c] for c in y_train_enc], dtype=np.float32)

print(f"Weights: {weights}")
print(f"sum_weights: {sample_weights.sum():.2f}") 

xgb_model = xgb.XGBClassifier(
    n_estimators          = 500,
    learning_rate         = 0.05,
    max_depth             = 5,
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    reg_alpha             = 0.1,
    reg_lambda            = 1.0,
    objective             = 'multi:softmax',
    num_class             = len(le.classes_),
    eval_metric           = 'mlogloss',
    early_stopping_rounds = 30,
    random_state          = 42,
    n_jobs                = -1
)

print(f"\nTraining XGBoost... Train: {len(X_train)} | Test: {len(X_test)}")
xgb_model.fit(
    X_train, y_train_enc,
    sample_weight = sample_weights,
    eval_set      = [(X_test, y_test_enc)],
    verbose       = 50
)

In [ ]:

xgb_test_preds  = le.inverse_transform(xgb_model.predict(X_test))
xgb_train_preds = le.inverse_transform(xgb_model.predict(X_train))

print("\n--- XGBoost Test Results ---")
print(classification_report(y_test, xgb_test_preds,
                             target_names=['Bigdown', 'Bigup', 'Stable']))

# Confusion Matrix
cm_xgb = confusion_matrix(y_test, xgb_test_preds,
                           labels=['Bigdown', 'Bigup', 'Stable'])
plt.figure(figsize=(8, 6))
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Bigdown', 'Bigup', 'Stable'],
            yticklabels=['Bigdown', 'Bigup', 'Stable'])
plt.title("XGBoost Confusion Matrix — Test")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.show()

# Feature Importance
feat_imp_xgb = pd.Series(xgb_model.feature_importances_, index=X_train.columns)
feat_imp_xgb.sort_values().plot(kind='barh', figsize=(8, 6), color='steelblue')
plt.title("XGBoost Feature Importance (Gain)")
plt.tight_layout()
plt.show()

print("--- XGBoost Train Results ---")
print(classification_report(y_train, xgb_train_preds,
                             target_names=['Bigdown', 'Bigup', 'Stable']))

In [ ]:

drop_cols = [
    'label', 'date', 'type',
    'down_days_7', 'taker_buy_volume',
    'z_score', 'taker_buy_ratio', 'taker_sell_ratio', 'negative_momentum',
]

X      = df.drop(columns=drop_cols)
y      = df['label']
groups = df['type']

le_logo = LabelEncoder().fit(y)

logo    = LeaveOneGroupOut()
results = []

for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups), 1):

    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

    test_coin = groups.iloc[test_idx].unique()[0]

    # Encode labels
    y_tr_enc = le_logo.transform(y_tr)
    y_te_enc = le_logo.transform(y_te)

    # Class weights per fold
    counts_fold  = Counter(y_tr_enc)
    total_fold   = sum(counts_fold.values())
    w_fold       = {c: total_fold / (len(counts_fold) * n) for c, n in counts_fold.items()}
    sw_fold      = np.array([w_fold[c] for c in y_tr_enc])

    # Train
    model_fold = xgb.XGBClassifier(
        n_estimators       = 500,
        learning_rate      = 0.05,
        max_depth          = 5,
        subsample          = 0.8,
        colsample_bytree   = 0.8,
        reg_alpha          = 0.1,
        reg_lambda         = 1.0,
        objective          = 'multi:softmax',
        num_class          = 3,
        eval_metric        = 'mlogloss',
        early_stopping_rounds = 30,
        random_state       = 42,
        n_jobs             = -1,
        use_label_encoder  = False
    )

    model_fold.fit(
        X_tr, y_tr_enc,
        sample_weight = sw_fold,
        eval_set      = [(X_te, y_te_enc)],
        verbose       = False
    )

    preds_enc = model_fold.predict(X_te)
    preds     = le_logo.inverse_transform(preds_enc)

    acc         = accuracy_score(y_te, preds)
    macro_f1    = f1_score(y_te, preds, average='macro')
    weighted_f1 = f1_score(y_te, preds, average='weighted')

    results.append({
        'test_coin'  : test_coin,
        'accuracy'   : acc,
        'macro_f1'   : macro_f1,
        'weighted_f1': weighted_f1
    })

    print(f"\n===== Fold {fold} | Test coin: {test_coin} =====")
    print(classification_report(y_te, preds,
                                 target_names=['Bigdown', 'Bigup', 'Stable']))

# --- Summary ---
results_df = pd.DataFrame(results)
print("\n", results_df)
print("\nAverage Results:")
print(results_df[['accuracy', 'macro_f1', 'weighted_f1']].mean())